# Phase 1: AI Foundations & LLM Fundamentals
## Day 4: Decorators for Token Usage & Latency

### Core Theory (Just-in-Time)
Welcome to Day 4. In production AI Engineering, **observability** is non-negotiable. Two of the most critical metrics you must track for every LLM call are:
1. **Latency:** How long the LLM takes to respond. This directly impacts user experience (UX) and system throughput.
2. **Token Usage:** How many tokens (prompt + completion) were consumed. This directly impacts your unit economics and API rate limits.

To monitor these metrics cleanly without cluttering your core business logic, we use **Python Decorators**. A decorator is a structural design pattern (often known as a wrapper) that allows you to add cross-cutting concerns—like logging, timing, and token counting—to existing functions dynamically. 

When integrating with LangChain, tracking these metrics can be done easily via standard callbacks (like `get_openai_callback`). By packaging this tracking into a reusable decorator, we ensure every LLM invocation across our application is consistently monitored.

### Code Implementation
Below is a tiered progression (Basic, Medium, Advanced) showing how to build decorators for token and latency measurement.

#### Basic: Simple Execution Timer
Isolates the core concept: wrapping a function to measure its execution time.

In [1]:
import time
from typing import Callable, Any

def time_execution(func: Callable) -> Callable:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"[{func.__name__}] Execution time: {end_time - start_time:.4f}s")
        return result
    return wrapper

@time_execution
def dummy_llm_call(prompt: str) -> str:
    time.sleep(0.1) # Simulate network delay
    return f"Response to: {prompt}"

dummy_llm_call("Hello World")

[dummy_llm_call] Execution time: 0.1002s


'Response to: Hello World'

#### Medium: Adding Token Tracking with LangChain
Shows how multiple concepts interact: using `functools.wraps` and `get_openai_callback` to measure tokens.

In [2]:
import time
from functools import wraps
from typing import Callable, Any
from langchain_community.callbacks.manager import get_openai_callback
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def track_metrics(func: Callable) -> Callable:
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.perf_counter()
        
        with get_openai_callback() as cb:
            try:
                result = func(*args, **kwargs)
            except Exception as e:
                print(f"Error: {e}")
                return "Fallback response"
            
        latency = time.perf_counter() - start_time
        print(f"[{func.__name__}] Latency: {latency:.4f}s | Tokens: {cb.total_tokens} | Cost: ${cb.total_cost:.6f}")
        return result
    return wrapper

@track_metrics
def summarize_text(text: str) -> str:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    messages = [HumanMessage(content=f"Summarize: {text}")]
    response = llm.invoke(messages)
    return str(response.content)

# Need valid API key to work
summarize_text("Decorators wrap functions to add behavior.")

/tmp/ipykernel_85870/3638250559.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks.manager import get_openai_callback


Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


'Fallback response'

#### Advanced: Production-Grade Async Tracking
Production-grade implementation with strict type hinting, docstrings, structured logging, error handling, and async support.

In [3]:
import asyncio
import json
import time
import logging
from functools import wraps
from typing import Callable, Any
from langchain_community.callbacks.manager import get_openai_callback
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

def async_track_metrics(func: Callable) -> Callable:
    """
    Production-grade async decorator to measure latency, token usage, and cost.
    Outputs structured JSON logs for observability platforms.
    """
    @wraps(func)
    async def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.perf_counter()
        
        try:
            with get_openai_callback() as cb:
                result = await func(*args, **kwargs)
        except Exception as e:
            logger.error(json.dumps({"event": "llm_error", "error": str(e)}))
            raise
        finally:
            latency = time.perf_counter() - start_time
            # Safely log metrics even if call fails, if callback variables exist
            if 'cb' in locals():
                log_data = {
                    "event": "llm_call",
                    "function_name": func.__name__,
                    "latency_seconds": round(latency, 4),
                    "total_tokens": cb.total_tokens,
                    "prompt_tokens": cb.prompt_tokens,
                    "completion_tokens": cb.completion_tokens,
                    "cost_usd": round(cb.total_cost, 6)
                }
                logger.info(json.dumps(log_data))
            
        return result
    return wrapper

@async_track_metrics
async def async_summarize_text(text: str) -> str:
    """
    Asynchronously generate a summary using ChatOpenAI.
    """
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    messages = [HumanMessage(content=f"Summarize: {text}")]
    try:
        response = await llm.ainvoke(messages)
        return str(response.content)
    except Exception:
        # Fallback to prevent Notebook failure without key
        return "Fallback summary (no API key)"

# Run the async function
# In a regular script, use asyncio.run(async_summarize_text(...))
# In Jupyter notebooks, we can await directly:
await async_summarize_text("Async execution is essential for high-throughput AI services.")

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


{"event": "llm_call", "function_name": "async_summarize_text", "latency_seconds": 0.2809, "total_tokens": 0, "prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}


'Fallback summary (no API key)'

### Practical Lab / Homework
Your actionable coding task for today:

1. **Custom LLM Tracker**: Create a decorator named `@custom_llm_tracker`.
   - Instead of using `get_openai_callback`, assume you are tracking an LLM that returns a dictionary like `{"response": "...", "usage": {"total_tokens": 50}}`.
2. **Implement**: Write the decorator to extract this `usage` dictionary from the return value and print the total tokens and latency.
3. **Test**: Apply it to a dummy function that simulates returning this dictionary structure.

*Write your code in the cell below to complete the lab.*

In [4]:
# Write your Lab / Homework implementation here
import time
from functools import wraps
from typing import Callable, Any, Dict

def custom_llm_tracker(func: Callable) -> Callable:
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        latency = time.perf_counter() - start_time
        
        # Extract usage
        usage = result.get("usage", {})
        tokens = usage.get("total_tokens", 0)
        
        print(f"[{func.__name__}] Latency: {latency:.4f}s | Total Tokens: {tokens}")
        return result
    return wrapper

@custom_llm_tracker
def dummy_llm_call_with_usage(prompt: str) -> Dict[str, Any]:
    time.sleep(0.05)
    return {
        "response": f"Generated text for {prompt}",
        "usage": {"total_tokens": 42}
    }

dummy_llm_call_with_usage("Hello world!")


[dummy_llm_call_with_usage] Latency: 0.0503s | Total Tokens: 42


{'response': 'Generated text for Hello world!', 'usage': {'total_tokens': 42}}

### Common Pitfalls
Here is what typically goes wrong in production when measuring these metrics:

1. **Sync Decorators on Async Functions:** Applying a standard wrapper to an `async` function returns a coroutine object immediately without waiting for execution. This logs zero execution time and counts zero tokens. You must use `async def wrapper(*args, **kwargs):` and `await func(...)` for async code.
2. **Streaming Tokens:** By default, OpenAI API responses that use `stream=True` often do not return token counts in standard chunks. You must explicitly request token usage by setting stream options (e.g., `stream_options={"include_usage": True}`) and parsing the final chunks carefully. The default `get_openai_callback()` might miss streaming tokens if not configured properly.
3. **Using `time.time()` instead of `time.perf_counter()`:** `time.time()` is subject to system clock updates (NTP synchronization) which can skew latency measurements. Always use `time.perf_counter()` for precision profiling.
4. **Losing Function Signatures:** Forgetting to use `@wraps(func)` means your function loses its `__name__` and `__doc__` attributes, replacing them with the wrapper's name. This heavily breaks routing libraries like FastAPI or Streamlit caching mechanisms.

### Reference Links
1. [Python `functools.wraps` Documentation](https://docs.python.org/3/library/functools.html#functools.wraps)
2. [LangChain Callbacks Documentation](https://python.langchain.com/docs/modules/callbacks/)
3. [Python `time.perf_counter` Documentation](https://docs.python.org/3/library/time.html#time.perf_counter)